<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 2 - RAG and Prompt Engineering to Extract Real-World Phenotypes (EPI 264)

This notebook introduces **prompt basics** using a local LLM (via Ollama + LangChain), and then applies prompts to a subset of **real (de-identified) clinical notes** prepared in Lab 1.

## Learning goals
By the end of this notebook, you will be able to:
- Send **system + user** messages to an LLM
- Use **prompt templates** (parameterized prompts) for repeatable workflows
- Apply a structured extraction prompt to real notes from our Lab 6 cohort

> Notes: We are not building embeddings or retrieval yet. This notebook is about **prompting** and **structured extraction**.

In [9]:
# -----------------------------------------------------------
# 1. Load the Ollama Model
# -----------------------------------------------------------
# This cell loads a local Large Language Model (LLM) using LangChain's
# `ChatOllama` wrapper. Make sure the model has been pulled via Ollama CLI before use.

# To explore available models, visit:
# - Ollama Library: https://ollama.com/library
# - Hugging Face (for embeddings and additional models): https://huggingface.co/models

from langchain_ollama import ChatOllama

# Define the model name — make sure this model is already downloaded using:
#   ollama pull deepseek-v2:16b
model_name = "qwen2"  # Alternatives: "qwen2", "llama3", "gpt-oss:20b" etc.

# Initialize the model with LangChain
model = ChatOllama(model=model_name)

print(f"✅ Model '{model_name}' is loaded and ready to use.")


✅ Model 'qwen2' is loaded and ready to use.


## 2. Basic Prompt Interaction (System + User Messages)

In this section, you will run a simple clinical query to confirm your local model is working.

**Key idea:**
- The **system message** sets role + behavior (“how to answer”).
- The **user message** asks the question (“what to answer”).

<img src="./images/basic_prompt.png" alt="i2b2 Logo" width="900">


In [10]:
# -----------------------------------------------------------
# 2. Run a Simple Clinical Query with System + User Prompts
# -----------------------------------------------------------
# This tests the model by asking a simple clinical question using structured messages.

from langchain_core.messages import HumanMessage, SystemMessage
from IPython.display import display, Markdown



messages = [
    SystemMessage(content=("""
        You are a knowledgeable medical provider.
        Provide clear, evidence-based explanations about a medical conditions.
    """
    )),
    HumanMessage(content="What is dementia? What are its common symptoms and treatments?")
]

# Run inference
response = model.invoke(messages)

# Display result
display(Markdown("### Model Response:"))
display(Markdown(response.content))


### Model Response:

Dementia is a term used to describe a decline in mental abilities that affects an individual's daily life. It involves a broad range of disorders characterized by a gradual loss of memory, cognitive skills (such as reasoning, judgment, and problem-solving), language proficiency, or the ability to understand time and place. These changes are not due to other medical conditions but are associated with aging.

### Common Types of Dementia
1. **Alzheimer's Disease**: The most common form of dementia, affecting up to 60-80% of individuals diagnosed with dementia.
2. **Vascular Dementia**: Results from damage or loss in the brain due to insufficient blood flow and may occur after a stroke or chronic conditions like heart disease.
3. **Lewy Body Dementia**: Characterized by fluctuations in mental abilities, sleep disorders, and hallucinations.
4. **Frontotemporal Dementia**: Affects behavior and personality rather than memory, affecting the frontal and temporal lobes of the brain.

### Common Symptoms
The symptoms vary depending on which part of the brain is affected but generally include:
1. **Memory Loss**: Difficulty remembering recent events or forgetting where things were placed.
2. **Trouble with Language**: Struggling to find words or using them inappropriately.
3. **Problems with Reasoning and Planning**: Difficulty making decisions, planning tasks, or managing time.
4. **Mood Changes**: Sudden mood swings, depression, anxiety, or apathy.
5. **Difficulty with Everyday Tasks**: Challenges with cooking, cleaning, driving, or handling money.

### Causes
Dementia is often caused by diseases that affect the brain such as Alzheimer's disease and vascular dementia. It can be due to a single disease or multiple diseases affecting different parts of the brain simultaneously.

### Treatments
1. **Medication**: Medications like acetylcholinesterase inhibitors (donepezil, rivastigmine) are used in Alzheimer’s disease to help manage symptoms by improving memory and cognitive function.
2. **Behavioral Management**: For managing behavioral symptoms such as agitation or depression, strategies might include cognitive-behavioral therapy for dementia patients or pharmacological treatment.
3. **Supportive Care**: Ensuring a safe environment with routines that are familiar helps reduce confusion and frustration.
4. **Occupational Therapy**: Can assist in maintaining daily living skills by providing practical exercises tailored to the individual's needs.

### Prevention
While dementia cannot be completely prevented, certain lifestyle choices can help reduce the risk:
1. **Maintain Mental Activity**: Engage in regular brain-stimulating activities like reading, puzzles, and learning.
2. **Healthy Diet**: A diet rich in fruits, vegetables, whole grains, lean protein, and healthy fats (like omega-3 fatty acids) may lower dementia risk.
3. **Regular Exercise**: Physical activity helps maintain cognitive function and reduces the risk of developing dementia.
4. **Manage Chronic Diseases**: Controlling high blood pressure, diabetes, and cholesterol levels can help prevent or delay the onset of dementia.

### Conclusion
Dementia is a complex condition affecting various aspects of life for individuals and their families. Early detection and management are crucial, focusing on symptom control, maintaining quality of life, and providing emotional support.

## 3. Prompt Templates (Reusable Prompts)

Hard-coding prompts works for one-off queries, but research workflows need **repeatable prompts**.

In this section, you will use `ChatPromptTemplate` to:
- Define a prompt once (with placeholders)
- Reuse it across conditions / patient contexts
- Standardize outputs across runs (important for reproducibility)

<img src="./images/prompt_template.png" alt="Prompt Template" width="900">


In [11]:
# -----------------------------------------------------------
# 3.1. Create a Reusable Prompt Template (Dynamic Querying)
# -----------------------------------------------------------
# This cell demonstrates how to build a dynamic prompt template using placeholders
# for different medical conditions and patient profiles. The prompt is populated
# with variables at runtime and sent to the LLM for inference.

from langchain_core.prompts import ChatPromptTemplate

# Define input variables
patient_type = "5-year old child"
disease = "dementia"

# Define a role-based prompt using variable placeholders
messages = [
    ("system", """
You are a knowledgeable medical provider. Provide clear, accurate, and evidence-based explanations appropriate for a {patient_type} patient.

Guidelines:
- Use plain language appropriate to the patient type.
- Avoid unnecessary medical jargon.
"""),

    ("human", """
What is {disease}? What are its common symptoms and treatments?
""")
]

# Create a prompt template with dynamic inputs
prompt_template = ChatPromptTemplate.from_messages(messages)

# Fill the template with actual values
prompt = prompt_template.invoke({
    "patient_type": patient_type,
    "disease": disease
})

# Run the model with the constructed prompt
response = model.invoke(prompt)

# Display the generated answer
display(Markdown("### AI Response"))
display(Markdown(response.content))


### AI Response

Dementia is like losing some of your brain's powers, making it harder for you to do things that were once easy. Imagine trying to find your favorite toy but can't remember where it was last played with – that’s a bit like what someone with dementia might experience.

Common symptoms include:

1. **Forgetfulness**: Sometimes people forget where they put their toys or might not remember if they already brushed their teeth before going outside.
2. **Confusion**: They might get confused about the day of the week, thinking it's a different day than it actually is. It’s like when you think Halloween comes in July instead of October!
3. **Difficulty with Words**: Sometimes people have trouble finding words or may use them wrong. It’s kind of like accidentally calling your mom "Dad" or forgetting what to call the kitchen helper, the refrigerator.
4. **Trouble with Planning and Doing Tasks**: Completing tasks that were simple before might become very hard, like putting away toys in the right place.

Treatments for dementia often include:

1. **Medications**: Sometimes doctors give special medicines to help slow down how fast the brain gets worse.
2. **Support**: Having people who can help them remember things and do tasks they find difficult becomes really important.
3. **Safe Environment**: Keeping their room or home where they live safe, with nothing sharp that could hurt them if they bump into it, helps a lot.

It's also really helpful for everyone to understand dementia better so we can be patient and gentle as someone's brain changes due to this condition.

In [12]:
# -----------------------------------------------------------
# 3.2. Prompt Template for Dementia Phenotype Extraction (Yes/No)
# -----------------------------------------------------------
# Goal:
# Demonstrate that an LLM can extract a phenotype from unstructured text.
# -----------------------------------------------------------
from langchain_core.prompts import ChatPromptTemplate

messages_notes = [
    ("system", """
You are an advanced clinical phenotyping assistant.
Your task is to extract key clinical details from unstructured notes in a clear, structured format.

Rules:
- Use ONLY information explicitly present in the note. Do NOT infer or invent.
- If the note is administrative/non-clinical (e.g., 'created in error', 'dictated', 'reconcile meds'), still output sections 1–4, but keep them minimal.
- Do NOT include meta statements about redaction, privacy, dates shifted, or removed identifiers.
"""),

    ("human", """
Analyze the following clinical note:

{patient_note}

Extract the following sections using numbered headings and bullet points:
1. Patient demographics (only what is stated)
2. Chief complaints / reason for visit
3. Current medications (only those explicitly listed)
4. Dementia phenotype (Yes/No)
   - Answer 'Yes' ONLY if dementia is documented (e.g., 'dementia', 'Alzheimer',
     'vascular dementia', 'Lewy body dementia') OR clearly stated as a prior diagnosis.
   - If dementia is not explicitly mentioned, answer 'No'. Do NOT infer from memory complaints alone.
   - Under section 4, include these sub-bullets (one per line):
     • Phenotype: Yes/No
     • Rationale: 1–2 sentences grounded ONLY in the evidence
     • Confidence: low/medium/high

Important: Keep the output cleanly separated by sections 1–4
(do not combine evidence/rationale/confidence into one line).
""")
]

prompt_template_notes = ChatPromptTemplate.from_messages(messages_notes)

print(">> Prompt created and ready to use.")

>> Prompt created and ready to use.


## 4. Apply Prompts to Real Notes from the Lab 1 Cohort

Now we will load:
- the cleaned notes dataset (deduplicated + restricted to the chosen 2-year window in Lab 1)
- the structured dementia flags + physician gold label

We will:
1. Merge notes with patient-level labels
2. Inspect notes for one patient
3. Run a structured extraction prompt on a selected note

> Reminder: These notes are de-identified synthetic narratives derived from original notes, designed for teaching and downstream RAG.

In [13]:
# -----------------------------------------------------------
# 4.1. Load Cleaned Notes and Structured Evaluation Dataset
# -----------------------------------------------------------
# From Lab 6:
# - lab6_clean_notes_baseline.parquet
# - lab6_structured_dementia_eval.csv
# -----------------------------------------------------------

# Purpose:
# Load the data into a pandas DataFrame and inspect the structure before decoding.

from pathlib import Path
import pandas as pd

filepath = Path("data_files")

notes = pd.read_parquet(filepath / "lab1_clean_notes_baseline.parquet")
eval_df = pd.read_csv(filepath / "lab1_structured_dementia_eval.csv", dtype=str)

print("Notes shape:", notes.shape)
print("Eval dataset shape:", eval_df.shape)

display(notes.head(10))
display(eval_df.head(10))


Notes shape: (2507, 7)
Eval dataset shape: (90, 6)


,patient_num,visit_id,note_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,note_txt_deid
0,H120513333,6297755287,2419560135,2305518755,Progress Notes,2017-01-01 16:08:00,Physical therapy progress note for visit #2. T...
1,H111336931,6332144391,2829946925,2712892487,Progress Notes,2017-01-01 17:11:00,The patient was seen for follow-up regarding s...
2,H120897999,6358002173,3208794021,3103021255,Progress Notes,2017-01-01 18:30:00,This is a progress note for a 69-year-old woma...
3,H120897999,6358002173,3210662987,3104938425,Assessment & Plan Note,2017-01-02 17:58:00,Assessment & Plan:\n\nThe patient has a histor...
4,H122074591,6270644869,2578637669,2455733125,Progress Notes,2017-01-03 13:56:00,This is an update on an 85-year-old man with a...
5,H122355386,6283144543,2149720299,2050365293,Progress Notes,2017-01-03 17:32:00,A message was left for the patient requesting ...
6,H120189926,6346141193,2984884099,2872365651,Assessment & Plan Note,2017-01-03 18:39:00,The patient was evaluated for blood pressure m...
7,H120189926,6346141193,2984887107,2872410355,Progress Notes,2017-01-03 18:40:00,The patient was seen for evaluation and manage...
8,H113566534,6313898129,2700402247,2579340567,Progress Notes,2017-01-04 14:40:00,Interval History:\nThe patient returned for fo...
9,H113383484,6341517751,3104765319,2995799951,Progress Notes,2017-01-04 17:31:00,The patient presented for a follow-up evaluati...


,patient_num,dx_dementia_flag,med_dementia_flag,baseline_dementia,gold_diagnosis,gold_dementia
0,H122355386,False,False,False,MCI vs. dementia,False
1,H112639340,False,False,False,No MCI or dementia diagnosis (normal),False
2,H117728724,False,False,False,No MCI or dementia diagnosis (normal),False
3,H116273351,False,False,False,No MCI or dementia diagnosis (normal),False
4,H113296266,False,False,False,No MCI or dementia diagnosis (normal),False
5,H112117815,False,True,True,Dementia,True
6,H115421676,False,False,False,No MCI or dementia diagnosis (normal),False
7,H115051235,False,False,False,No MCI or dementia diagnosis (normal),False
8,H111969872,False,False,False,Dementia,True
9,H119661286,False,False,False,No MCI or dementia diagnosis (normal),False


In [14]:
# -----------------------------------------------------------
# 4.2. Merge Notes with Structured & Gold Labels
# -----------------------------------------------------------

notes_eval = notes.merge(
    eval_df,
    on="patient_num",
    how="inner"
)

print("Merged dataset shape:", notes_eval.shape)
display(notes_eval.head(10))

Merged dataset shape: (2422, 12)


,patient_num,visit_id,note_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,note_txt_deid,dx_dementia_flag,med_dementia_flag,baseline_dementia,gold_diagnosis,gold_dementia
0,H120513333,6297755287,2419560135,2305518755,Progress Notes,2017-01-01 16:08:00,Physical therapy progress note for visit #2. T...,False,False,False,No MCI or dementia diagnosis (normal),False
1,H111336931,6332144391,2829946925,2712892487,Progress Notes,2017-01-01 17:11:00,The patient was seen for follow-up regarding s...,False,False,False,No MCI or dementia diagnosis (normal),False
2,H120897999,6358002173,3208794021,3103021255,Progress Notes,2017-01-01 18:30:00,This is a progress note for a 69-year-old woma...,False,False,False,No MCI or dementia diagnosis (normal),False
3,H120897999,6358002173,3210662987,3104938425,Assessment & Plan Note,2017-01-02 17:58:00,Assessment & Plan:\n\nThe patient has a histor...,False,False,False,No MCI or dementia diagnosis (normal),False
4,H122074591,6270644869,2578637669,2455733125,Progress Notes,2017-01-03 13:56:00,This is an update on an 85-year-old man with a...,False,False,False,Dementia,True
5,H122355386,6283144543,2149720299,2050365293,Progress Notes,2017-01-03 17:32:00,A message was left for the patient requesting ...,False,False,False,MCI vs. dementia,False
6,H120189926,6346141193,2984884099,2872365651,Assessment & Plan Note,2017-01-03 18:39:00,The patient was evaluated for blood pressure m...,False,False,False,No MCI or dementia diagnosis (normal),False
7,H120189926,6346141193,2984887107,2872410355,Progress Notes,2017-01-03 18:40:00,The patient was seen for evaluation and manage...,False,False,False,No MCI or dementia diagnosis (normal),False
8,H113566534,6313898129,2700402247,2579340567,Progress Notes,2017-01-04 14:40:00,Interval History:\nThe patient returned for fo...,False,False,False,No MCI or dementia diagnosis (normal),False
9,H113383484,6341517751,3104765319,2995799951,Progress Notes,2017-01-04 17:31:00,The patient presented for a follow-up evaluati...,False,False,False,No MCI or dementia diagnosis (normal),False


In [15]:
# -----------------------------------------------------------
# 4.3. Inspect First 5 Notes for a Specific Patient
# -----------------------------------------------------------

from IPython.display import display, Markdown

sample_patient = "H122074591"

sample_notes = notes_eval[
    notes_eval["patient_num"] == sample_patient
].head(5)

print(f"Showing first {sample_notes.shape[0]} notes for patient {sample_patient}")

# Display metadata table
display(sample_notes[[
    "patient_num",
    "visit_id",
    "note_csn_id",
    "inpatient_note_type_dsc",
    "create_dts_shifted",
    "baseline_dementia",
    "gold_dementia"
]])

# Display text for first 10 notes
for _, row in sample_notes.iterrows():
    display(Markdown(f"""
---
### Note CSN ID: {row['note_csn_id']}
**Visit ID:** {row['visit_id']}
**Note Type:** {row['inpatient_note_type_dsc']}
**Date:** {row['create_dts_shifted']}
**Structured Dementia:** {row['baseline_dementia']}
**Gold Dementia:** {row['gold_dementia']}

---

{row['note_txt_deid']}
"""))

Showing first 5 notes for patient H122074591


,patient_num,visit_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,baseline_dementia,gold_dementia
4,H122074591,6270644869,2455733125,Progress Notes,2017-01-03 13:56:00,False,True
64,H122074591,6297519395,2507760869,Progress Notes,2017-01-24 16:25:00,False,True
274,H122074591,6325821229,2650311453,Progress Notes,2017-04-07 15:35:00,False,True
545,H122074591,6326650057,2857997493,Progress Notes,2017-07-06 15:08:00,False,True
887,H122074591,6367433259,3065515755,ED Notes,2017-09-29 21:53:00,False,True



---
### Note CSN ID: 2455733125
**Visit ID:** 6270644869
**Note Type:** Progress Notes
**Date:** 2017-01-03 13:56:00
**Structured Dementia:** False
**Gold Dementia:** True

---

This is an update on an 85-year-old man with a history of persistent atrial fibrillation, hypertension, hypothyroidism, bradycardia, asthma, severe mitral regurgitation, prior cardiac arrest due to proarrhythmic effects of Propafenone, dementia, prior squamous skin carcinoma, and lactose intolerance. He has been on a rate control and anticoagulation strategy for atrial fibrillation since the late 1990s, with several past DC cardioversions. Currently, he is clinically stable, remains in atrial fibrillation, and is tolerating rivaroxaban well, with no reported bleeding or neurologic events. Creatinine clearance is 61 mL/min based on a recent creatinine value, and current rivaroxaban dosing remains appropriate.

Mitral regurgitation is severe, confirmed by echocardiogram findings from November 2014, showing marked thickening and flail of the posterior mitral leaflet; the patient and family have declined surgical intervention given overall age and comorbidities. He is euvolemic on exam and reports minimal symptoms, mainly leg fatigue with exertion. He denies chest pain, worsening shortness of breath, palpitations, syncope, near syncope, orthostasis, or claudication. Appetite and weight are stable, with prior fluctuations likely related to hypothyroidism. He sleeps on two pillows without symptoms suggestive of obstructive sleep apnea.

A mechanical fall and right hip fracture occurred two years ago, for which he underwent successful surgical repair. He is able to ambulate several minutes on flat surfaces without cardiovascular symptoms. Blood pressure on recent visit was 118/60 mmHg, pulse 56, BMI 28.3. Physical exam reveals no acute distress, clear lung fields, trace pedal edema, a holosystolic murmur at the apex, and diminished peripheral pulses.

Labs from July 11, 2016 show total cholesterol 206, HDL 42, LDL 134, triglycerides 148, and TSH elevated at 25.84 uIU/mL (reference 0.40–5.00). TSH from May 25, 2015 was also elevated at 8.610 uIU/mL. Echocardiogram findings include marked left atrial and right atrial enlargement, normal left ventricular size and function, estimated ejection fraction 79%, symmetric LV hypertrophy, severe mitral regurgitation, trace aortic insufficiency, and mild tricuspid and pulmonary insufficiency. Holter monitoring from November 2014 showed a baseline of atrial fibrillation, average heart rate of 57 bpm, minimum 33 bpm, no reported symptoms, and no evidence of high grade AV block or long pauses.

Current medications include levothyroxine for hypothyroidism, lisinopril for hypertension, and rivaroxaban for atrial fibrillation. He does not use over-the-counter medications, herbal remedies, or illegal substances. Alcohol use has ceased. There is a suggestion of mild alcoholism in the past. He previously smoked for 20 years but quit over 25 years ago.

Social history: retired attorney, living with wife who suffers from mild dementia and orthopedic issues. Family history notable for maternal premature coronary artery disease and heart failure and paternal prostate cancer.

Assessment and Plan:
1. Persistent atrial fibrillation: Clinically stable, continue rate control and anticoagulation. Continue current rivaroxaban dosing. No evidence of bleeding/neurologic events.
2. Severe mitral regurgitation: No surgical intervention planned. Monitor for symptoms/signs of decompensation; patient euvolemic.
3. Bradycardia: No concerning symptoms; warned regarding potential future need for pacemaker.
4. Hypertension: Controlled on current regimen.
5. Hypothyroidism: Continue current levothyroxine regimen; will recheck labs.
6. Encourage regular exercise, weight maintenance, avoidance of alcohol. Advised to obtain influenza vaccine.
7. Primary care: Will establish care with new provider.

Follow-up is planned in six months. There is a prior diagnosis of dementia documented in his medical history.



---
### Note CSN ID: 2507760869
**Visit ID:** 6297519395
**Note Type:** Progress Notes
**Date:** 2017-01-24 16:25:00
**Structured Dementia:** False
**Gold Dementia:** True

---

An 85-year-old male presented for follow-up, accompanied by family members. Primary concerns addressed were atrial fibrillation, hypothyroidism, dementia, vitamin D deficiency, and cerumen impaction.

He continues to live with his spouse and receives daily assistance with medication management and personal care, including help from an aide. The patient did not report any new complaints. On review, right ear blockage was noted, while hearing was otherwise reported as satisfactory. He exercises occasionally at the gym and discussed his diet during the visit. Memory difficulties were noted, consistent with a prior diagnosis of dementia.

Current medication regimen includes levothyroxine, lisinopril, rivaroxaban, sertraline, and vitamin D supplementation. He is adherent to his medications and tolerating them well without side effects.

Vital signs recorded: blood pressure 114/62 mmHg (right arm, sitting), pulse 66, respiratory rate 18, height 1.803 meters, weight 95.3 kg, BMI 29.3.

Physical examination revealed the patient to be alert, oriented, and in no distress. Eye exam was normal. Right ear canal was blocked with cerumen; left ear was unremarkable. Neck was supple without lymphadenopathy or thyroid masses. Lungs were clear with good air entry and no adventitious sounds. Cardiac exam showed a normal rate with irregular rhythm and a 2/6 murmur. No edema, clubbing, or cyanosis was seen in the extremities, and skin exam was unremarkable.

Recent laboratory evaluation (12/4/2016):
- NT-proBNP: 944 pg/mL
- Sodium: 144 mmol/L
- Chloride: 109 mmol/L (mildly elevated)
- Potassium: 4.5 mmol/L
- CO2: 24 mmol/L
- BUN: 18 mg/dL
- Creatinine: 1.17 mg/dL
- Glucose: 103 mg/dL
- Calcium: 8.8 mg/dL
- eGFR: 59 mL/min/1.73m2
- Anion gap: 11 mmol/L
- Magnesium: 2.3 mg/dL

Plans were made to continue current management for atrial fibrillation and hypothyroidism, initiate or continue vitamin D supplementation at 2000 units daily, and schedule a return visit in about two months (around 3/27/2017) for further lab assessment (Hemoglobin A1c, lipid panel, TSH, 25-OH vitamin D, CBC, and comprehensive metabolic panel). Cerumen impaction on the right will be addressed as needed.

Allergies, medication, and relevant histories were updated in the chart. No acute issues at this time. The patient and family were advised to monitor for changes and follow up as planned.



---
### Note CSN ID: 2650311453
**Visit ID:** 6325821229
**Note Type:** Progress Notes
**Date:** 2017-04-07 15:35:00
**Structured Dementia:** False
**Gold Dementia:** True

---

Progress Note:

This is an 85-year-old male who presented for a 40-minute visit, with the majority of time spent coordinating care and providing counseling. The primary concern remains Alzheimer's disease, for which ongoing treatment is being continued. He is doing well with assistance at home and displays understanding and insight regarding his care decisions, including completion of the MOLST form expressing his clear preferences. His healthcare proxy and durable power of attorney supports these decisions.

He describes intermittent, sharp left chest pain occurring roughly once every three weeks when falling asleep. The pain lasts 15-30 minutes and resolves with position change, without associated shortness of breath, diaphoresis, nausea, or vomiting. A cardiologist deemed the pain benign at a previous evaluation in late October 2016 (shifted to mid-January 2017). He denies exertional chest pain.

On examination, he appears alert and oriented, well appearing, and in no acute distress. He participates thoughtfully in conversation, though it is noted that this reflects his best performance with reported periods of poorer mental functioning. Gait is slow and hesitant initially, improving with ambulation. Lungs are clear; the heart rhythm is irregular without murmurs or gallops. There is 1+ edema in both shins. Multiple actinic keratoses (treated with cryotherapy) are present on the arms and right hand.

Vital signs: BP 116/64 (sitting, left arm); pulse 75; respirations 18; SpO2 99%; height 1.803 m; weight 94.9 kg; BMI 29.18.

Recent laboratory studies (shifted to mid-April 2017) show:
- Sodium: 145 mmol/L
- Chloride: 106 mmol/L
- Potassium: 4.1 mmol/L
- CO2: 25 mmol/L
- BUN: 18 mg/dL
- Creatinine: 1.18 mg/dL
- Glucose: 112 mg/dL (mild fasting blood sugar elevation)
- Calcium: 8.9 mg/dL
- eGFR: 59 mL/min/1.73m2
- Anion gap: 14 mmol/L
- 25-OH Vitamin D: 16 ng/mL (deficiency)
- Hemoglobin A1c: 5.1% (pre-diabetes)
- TSH: 1.21 uIU/mL

Past labs from late January 2017 (shifted from early November 2016) show similar results, with vitamin D persistently low (11 ng/mL), mild elevation in fasting glucose, and stable thyroid function.

Currently prescribed medications include cholecalciferol (1000 units daily), levothyroxine, lisinopril, rivaroxaban, and sertraline. No current side effects reported and medications are being taken as directed.

Active Problems and Plans:
- Chronic atrial fibrillation: Stable; continue present management.
- Hypothyroidism: Continue existing therapy.
- Alzheimer's dementia: Continue current treatment; doing well with assistance.
- Actinic keratoses: Cryotherapy performed; monitor.
- Vitamin D deficiency: Increase supplementation to 3000 units daily; recheck in three months.
- Prediabetes: Mild; recheck labs in three months.

The MOLST form was completed with clear patient preferences recorded. Home support remains robust.

Review of systems is negative for headache, palpitations, shortness of breath, cough, gastrointestinal or urinary symptoms, rash, fever, or constitutional complaints.

The next follow-up is recommended in approximately three months (around early July 2017) for repeat laboratory studies and office evaluation.



---
### Note CSN ID: 2857997493
**Visit ID:** 6326650057
**Note Type:** Progress Notes
**Date:** 2017-07-06 15:08:00
**Structured Dementia:** False
**Gold Dementia:** True

---

Interval Progress Note

This is an 86-year-old male with moderate Alzheimer's dementia, vitamin D deficiency, hypothyroidism, and chronic atrial fibrillation. The patient is reportedly golfing, walking, maintaining his usual diet, and reading newspapers. He takes his medications as prescribed, including vitamin D supplementation and levothyroxine, with no noted side effects.

On examination, the patient appears alert, well, and in no acute distress. He remains oriented to basic information and is able to engage appropriately, though occasionally has difficulty understanding questions. Current vital signs are blood pressure 128/62, pulse 55, respiratory rate 18, oxygen saturation 98%, height 1.803 m, weight 94.4 kg, and BMI 29.05.

Laboratory data from 06/30/2017 show a persistently low 25-OH vitamin D (20 ng/mL), with other values including sodium 143 mmol/L, chloride 103 mmol/L, potassium 4.6 mmol/L, CO2 24 mmol/L, BUN 19 mg/dL, creatinine 1.21 mg/dL, glucose 79 mg/dL, calcium 9.0 mg/dL, eGFR 57 mL/min/1.73m2, anion gap 16 mmol/L, hemoglobin A1c 5.5%, and calculated mean blood glucose 111 mg/dL.

Active medications include cholecalciferol (vitamin D3) 4,000 units daily, lisinopril 10 mg daily, rivaroxaban 20 mg daily, sertraline 100 mg daily, and levothyroxine 150 mcg (8 tablets per week).

The patient denies headache, chest pain, palpitations, dyspnea, cough, nausea, vomiting, abdominal pain, bowel or urinary symptoms, rash, and fever.

Assessment and Plan:
- Alzheimer's dementia: Course stable; patient remains active; continue current management.
- Vitamin D deficiency: Levels remain low; continue vitamin D supplementation and monitor administration.
- Hypothyroidism: Stable on levothyroxine; monitor thyroid function.
- Chronic atrial fibrillation: Stable; continue anticoagulation with rivaroxaban.
- Weight is stable.

Follow-up is planned in approximately three months (around 10/5/2017) for laboratory studies and evaluation.



---
### Note CSN ID: 3065515755
**Visit ID:** 6367433259
**Note Type:** ED Notes
**Date:** 2017-09-29 21:53:00
**Structured Dementia:** False
**Gold Dementia:** True

---

The patient's daughter requested a copy of the CT scan.


In [16]:
# -----------------------------------------------------------
# 4.4. Analyze a Real Patient Note Using note_csn_id
# -----------------------------------------------------------

from IPython.display import display, Markdown

NOTE_CSN_ID = 2650311453  # <-- set this to any note_csn_id you want

# Pull the note row
note_row = notes_eval.loc[notes_eval["note_csn_id"] == NOTE_CSN_ID].iloc[0]

# Extract the note text
patient_note_text = note_row["note_txt_deid"]

# Fill prompt
filled_prompt = prompt_template_notes.invoke({"patient_note": patient_note_text})

# Run model
clinical_response = model.invoke(filled_prompt)

# Display context + output
display(Markdown(f"""
### Note Selected
- **Patient:** {note_row['patient_num']}
- **Note CSN ID:** {note_row['note_csn_id']}
- **Note Type:** {note_row['inpatient_note_type_dsc']}
"""))

display(Markdown("### Extracted Clinical Information"))
display(Markdown(clinical_response.content))


### Note Selected
- **Patient:** H122074591
- **Note CSN ID:** 2650311453
- **Note Type:** Progress Notes


### Extracted Clinical Information

### Patient Demographics
- Age: 85 years
- Gender: Male

### Chief Complaints / Reason for Visit
- Coordination of care and counseling
- Ongoing treatment for Alzheimer's disease
- Assistance at home required
- MOLST form completed expressing clear preferences
- Intermittent, sharp left chest pain occurring roughly once every three weeks when falling asleep

### Current Medications
- Cholecalciferol (1000 units daily)
- Levothyroxine (Dosing not specified)
- Lisinopril (Dosing not specified)
- Rivaroxaban (Dosing not specified)
- Sertraline (Dosing not specified)

### Dementia Phenotype
- **Yes**
  - **Rationale:** Alzheimer's disease is explicitly mentioned as the primary concern and ongoing treatment.
  - **Confidence:** High